# Customer Churn Prediction Analysis

This notebook demonstrates how to build and evaluate customer churn prediction models using machine learning techniques.

## Table of Contents
1. [Data Preparation](#data-preparation)
2. [Feature Engineering](#feature-engineering)
3. [Model Development](#model-development)
4. [Model Evaluation](#model-evaluation)
5. [Feature Importance Analysis](#feature-importance)
6. [Business Insights](#business-insights)

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('default')
sns.set_palette("husl")

# Import our custom modules
import sys
sys.path.append('../src')
from data_processor import DataProcessor
from rfm_analyzer import RFMAnalyzer
from churn_predictor import ChurnPredictor

## 1. Data Preparation {#data-preparation}

First, let's load and prepare our customer transaction data for churn analysis.

In [ ]:
# Initialize data processor
processor = DataProcessor()

# Generate sample transaction data
print("Generating sample transaction data...")
transaction_data = processor.create_sample_dataset(n_customers=2000, n_transactions=10000)

print(f"Dataset shape: {transaction_data.shape}")
print(f"Date range: {transaction_data['transaction_date'].min()} to {transaction_data['transaction_date'].max()}")
print(f"Number of unique customers: {transaction_data['customer_id'].nunique()}")

transaction_data.head()

In [ ]:
# Clean and process the data
clean_data = processor.clean_transaction_data(transaction_data)

# Create customer-level features
customer_features = processor.create_customer_features(clean_data)

print(f"Customer features shape: {customer_features.shape}")
print("\nFeature columns:")
for col in customer_features.columns:
    print(f"- {col}")

customer_features.head()

## 2. Feature Engineering {#feature-engineering}

Let's create additional features that might be predictive of churn behavior.

In [ ]:
# Initialize RFM analyzer to get RFM scores
rfm_analyzer = RFMAnalyzer()
rfm_scores = rfm_analyzer.calculate_rfm(clean_data)

# Merge RFM scores with customer features
customer_data = customer_features.merge(rfm_scores, on='customer_id', how='left')

print(f"Combined dataset shape: {customer_data.shape}")
customer_data.head()

In [ ]:
# Define churn based on recency (customers who haven't purchased in last 90 days)
CHURN_THRESHOLD_DAYS = 90
customer_data['is_churned'] = (customer_data['recency'] > CHURN_THRESHOLD_DAYS).astype(int)

# Check churn distribution
churn_distribution = customer_data['is_churned'].value_counts()
churn_rate = customer_data['is_churned'].mean()

print(f"Churn Distribution:")
print(f"Active customers: {churn_distribution[0]} ({(1-churn_rate)*100:.1f}%)")
print(f"Churned customers: {churn_distribution[1]} ({churn_rate*100:.1f}%)")
print(f"\nOverall churn rate: {churn_rate:.3f}")

In [ ]:
# Visualize churn distribution
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Churn Distribution', 'Churn Rate by Customer Segment'],
    specs=[[{'type': 'pie'}, {'type': 'bar'}]]
)

# Pie chart for overall churn distribution
fig.add_trace(
    go.Pie(
        labels=['Active', 'Churned'],
        values=[churn_distribution[0], churn_distribution[1]],
        hole=0.4,
        marker_colors=['#2E8B57', '#DC143C']
    ),
    row=1, col=1
)

# Get RFM segments for additional analysis
customer_segments = rfm_analyzer.segment_customers(rfm_scores)
segment_data = customer_data.merge(customer_segments[['customer_id', 'segment']], on='customer_id')

# Churn rate by segment
segment_churn = segment_data.groupby('segment')['is_churned'].agg(['count', 'sum', 'mean']).round(3)
segment_churn.columns = ['total_customers', 'churned_customers', 'churn_rate']

fig.add_trace(
    go.Bar(
        x=segment_churn.index,
        y=segment_churn['churn_rate'],
        marker_color='lightblue',
        text=segment_churn['churn_rate'],
        textposition='outside'
    ),
    row=1, col=2
)

fig.update_layout(height=400, showlegend=False, title_text="Customer Churn Analysis")
fig.show()

print("\nChurn Rate by Customer Segment:")
print(segment_churn.sort_values('churn_rate', ascending=False))

## 3. Model Development {#model-development}

Now let's build machine learning models to predict customer churn.

In [ ]:
# Prepare features for modeling
feature_columns = [
    'total_transactions', 'total_amount', 'avg_transaction_amount',
    'days_since_first_purchase', 'purchase_frequency',
    'recency', 'frequency', 'monetary',
    'r_score', 'f_score', 'm_score'
]

X = customer_data[feature_columns].fillna(0)
y = customer_data['is_churned']

print(f"Feature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nFeatures used for modeling:")
for i, col in enumerate(feature_columns, 1):
    print(f"{i:2d}. {col}")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"Training churn rate: {y_train.mean():.3f}")
print(f"Test churn rate: {y_test.mean():.3f}")

In [ ]:
# Scale features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss')
}

# Train and evaluate models
model_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Use scaled data for logistic regression, original for tree-based models
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    auc_score = roc_auc_score(y_test, y_pred_proba)
    
    # Cross-validation score
    if name == 'Logistic Regression':
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
    else:
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    
    model_results[name] = {
        'model': model,
        'auc_score': auc_score,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"AUC Score: {auc_score:.4f}")
    print(f"CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## 4. Model Evaluation {#model-evaluation}

Let's evaluate and compare the performance of our models.

In [ ]:
# Create model comparison dataframe
comparison_df = pd.DataFrame({
    'Model': list(model_results.keys()),
    'AUC Score': [results['auc_score'] for results in model_results.values()],
    'CV Mean': [results['cv_mean'] for results in model_results.values()],
    'CV Std': [results['cv_std'] for results in model_results.values()]
})

comparison_df = comparison_df.sort_values('AUC Score', ascending=False)
print("Model Performance Comparison:")
print(comparison_df.round(4))

# Visualize model comparison
fig = px.bar(
    comparison_df, 
    x='Model', 
    y='AUC Score',
    title='Model Performance Comparison (AUC Score)',
    color='AUC Score',
    color_continuous_scale='viridis'
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# ROC Curves for all models
fig = go.Figure()

for name, results in model_results.items():
    fpr, tpr, _ = roc_curve(y_test, results['probabilities'])
    auc_score = results['auc_score']
    
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'{name} (AUC = {auc_score:.3f})'
    ))

# Add diagonal line
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    line=dict(dash='dash', color='gray'),
    name='Random Classifier'
))

fig.update_layout(
    title='ROC Curves Comparison',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=500
)
fig.show()

In [ ]:
# Detailed evaluation for best model
best_model_name = comparison_df.iloc[0]['Model']
best_model_results = model_results[best_model_name]

print(f"Detailed Evaluation for {best_model_name}:")
print("=" * 50)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, best_model_results['predictions']))

# Confusion matrix
cm = confusion_matrix(y_test, best_model_results['predictions'])
print("\nConfusion Matrix:")
print(cm)

# Visualize confusion matrix
fig = px.imshow(
    cm,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=['Not Churned', 'Churned'],
    y=['Not Churned', 'Churned'],
    text_auto=True,
    title=f'Confusion Matrix - {best_model_name}'
)
fig.show()

## 5. Feature Importance Analysis {#feature-importance}

Understanding which features are most important for predicting churn.

In [ ]:
# Feature importance for tree-based models
if best_model_name in ['Random Forest', 'XGBoost']:
    feature_importance = best_model_results['model'].feature_importances_
    
    importance_df = pd.DataFrame({
        'feature': feature_columns,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"Feature Importance ({best_model_name}):")
    print(importance_df.round(4))
    
    # Visualize feature importance
    fig = px.bar(
        importance_df.head(10),
        x='importance',
        y='feature',
        orientation='h',
        title=f'Top 10 Feature Importance - {best_model_name}',
        color='importance',
        color_continuous_scale='viridis'
    )
    fig.update_layout(height=500)
    fig.show()

# For logistic regression, show coefficients
elif best_model_name == 'Logistic Regression':
    coefficients = best_model_results['model'].coef_[0]
    
    coef_df = pd.DataFrame({
        'feature': feature_columns,
        'coefficient': coefficients,
        'abs_coefficient': np.abs(coefficients)
    }).sort_values('abs_coefficient', ascending=False)
    
    print(f"Feature Coefficients ({best_model_name}):")
    print(coef_df.round(4))
    
    # Visualize coefficients
    fig = px.bar(
        coef_df.head(10),
        x='coefficient',
        y='feature',
        orientation='h',
        title=f'Feature Coefficients - {best_model_name}',
        color='coefficient',
        color_continuous_scale='RdBu'
    )
    fig.update_layout(height=500)
    fig.show()

In [ ]:
# Analyze feature distributions by churn status
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Recency Distribution', 'Frequency Distribution', 
                   'Monetary Distribution', 'Transaction Amount Distribution']
)

# Recency
for churn_status, label in [(0, 'Active'), (1, 'Churned')]:
    data = customer_data[customer_data['is_churned'] == churn_status]['recency']
    fig.add_trace(
        go.Histogram(x=data, name=f'{label}', opacity=0.7, nbinsx=30),
        row=1, col=1
    )

# Frequency
for churn_status, label in [(0, 'Active'), (1, 'Churned')]:
    data = customer_data[customer_data['is_churned'] == churn_status]['frequency']
    fig.add_trace(
        go.Histogram(x=data, name=f'{label}', opacity=0.7, nbinsx=20, showlegend=False),
        row=1, col=2
    )

# Monetary
for churn_status, label in [(0, 'Active'), (1, 'Churned')]:
    data = customer_data[customer_data['is_churned'] == churn_status]['monetary']
    fig.add_trace(
        go.Histogram(x=data, name=f'{label}', opacity=0.7, nbinsx=30, showlegend=False),
        row=2, col=1
    )

# Average transaction amount
for churn_status, label in [(0, 'Active'), (1, 'Churned')]:
    data = customer_data[customer_data['is_churned'] == churn_status]['avg_transaction_amount']
    fig.add_trace(
        go.Histogram(x=data, name=f'{label}', opacity=0.7, nbinsx=30, showlegend=False),
        row=2, col=2
    )

fig.update_layout(height=600, title_text="Feature Distributions by Churn Status")
fig.show()

## 6. Business Insights {#business-insights}

Let's derive actionable business insights from our churn prediction model.

In [ ]:
# Predict churn probabilities for all customers
if best_model_name == 'Logistic Regression':
    X_scaled = scaler.transform(X)
    churn_probabilities = best_model_results['model'].predict_proba(X_scaled)[:, 1]
else:
    churn_probabilities = best_model_results['model'].predict_proba(X)[:, 1]

# Add predictions to customer data
customer_data['churn_probability'] = churn_probabilities
customer_data['risk_category'] = pd.cut(
    churn_probabilities,
    bins=[0, 0.3, 0.7, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

# Risk category distribution
risk_distribution = customer_data['risk_category'].value_counts()
print("Customer Risk Distribution:")
for category, count in risk_distribution.items():
    pct = (count / len(customer_data)) * 100
    print(f"{category}: {count} customers ({pct:.1f}%)")

# Visualize risk distribution
fig = px.pie(
    values=risk_distribution.values,
    names=risk_distribution.index,
    title='Customer Churn Risk Distribution',
    color_discrete_map={
        'Low Risk': '#2E8B57',
        'Medium Risk': '#FFD700',
        'High Risk': '#DC143C'
    }
)
fig.show()

In [ ]:
# Analyze characteristics of high-risk customers
high_risk_customers = customer_data[customer_data['risk_category'] == 'High Risk']
low_risk_customers = customer_data[customer_data['risk_category'] == 'Low Risk']

print("Characteristics Comparison:")
print("=" * 50)

comparison_metrics = [
    'recency', 'frequency', 'monetary', 'total_transactions',
    'avg_transaction_amount', 'days_since_first_purchase'
]

comparison_df = pd.DataFrame({
    'Metric': comparison_metrics,
    'High Risk (Mean)': [high_risk_customers[col].mean() for col in comparison_metrics],
    'Low Risk (Mean)': [low_risk_customers[col].mean() for col in comparison_metrics]
})

comparison_df['Difference'] = comparison_df['High Risk (Mean)'] - comparison_df['Low Risk (Mean)']
comparison_df['Ratio'] = comparison_df['High Risk (Mean)'] / comparison_df['Low Risk (Mean)']

print(comparison_df.round(2))

In [ ]:
# Business recommendations based on risk categories
recommendations = {
    'High Risk': [
        'Immediate intervention required',
        'Personal outreach by customer success team',
        'Offer significant discounts or incentives',
        'Conduct exit interviews to understand issues',
        'Provide premium support and service'
    ],
    'Medium Risk': [
        'Proactive engagement campaigns',
        'Send personalized product recommendations',
        'Offer loyalty rewards or points',
        'Regular check-ins via email or phone',
        'Invite to exclusive events or webinars'
    ],
    'Low Risk': [
        'Maintain current engagement level',
        'Focus on upselling and cross-selling',
        'Collect referrals and testimonials',
        'Use as brand ambassadors',
        'Regular satisfaction surveys'
    ]
}

print("Business Recommendations by Risk Category:")
print("=" * 50)

for risk_level, actions in recommendations.items():
    count = risk_distribution[risk_level]
    print(f"\n{risk_level} Customers ({count} customers):")
    for i, action in enumerate(actions, 1):
        print(f"  {i}. {action}")

In [ ]:
# Calculate potential revenue impact
# Assume average customer value and retention costs
avg_customer_value = customer_data['monetary'].mean()
retention_cost_pct = 0.2  # 20% of customer value

high_risk_count = len(high_risk_customers)
high_risk_value = high_risk_customers['monetary'].sum()

# Potential revenue at risk
revenue_at_risk = high_risk_value
retention_investment = high_risk_count * avg_customer_value * retention_cost_pct

print(f"Revenue Impact Analysis:")
print(f"=" * 30)
print(f"High-risk customers: {high_risk_count}")
print(f"Revenue at risk: ${revenue_at_risk:,.2f}")
print(f"Recommended retention investment: ${retention_investment:,.2f}")
print(f"Potential ROI if 50% retained: {((revenue_at_risk * 0.5 - retention_investment) / retention_investment * 100):.1f}%")

# Top high-risk customers by value
top_risk_customers = high_risk_customers.nlargest(10, 'monetary')[[
    'customer_id', 'monetary', 'churn_probability', 'recency', 'frequency'
]]

print(f"\nTop 10 High-Risk Customers by Value:")
print(top_risk_customers.round(3))

## Summary

### Key Findings:

1. **Model Performance**: Our best model achieved an AUC score of {best_model_results['auc_score']:.3f}, indicating good predictive performance.

2. **Important Features**: The most important features for predicting churn are:
   - Recency (days since last purchase)
   - Purchase frequency
   - Customer monetary value
   - RFM scores

3. **Risk Distribution**: 
   - {risk_distribution['Low Risk']} customers are low risk
   - {risk_distribution['Medium Risk']} customers are medium risk  
   - {risk_distribution['High Risk']} customers are high risk

4. **Business Impact**: 
   - ${revenue_at_risk:,.0f} in revenue is at risk from high-risk customers
   - Targeted retention campaigns could generate significant ROI

### Next Steps:

1. **Deploy Model**: Integrate the churn prediction model into production systems
2. **Monitor Performance**: Track model accuracy and retrain regularly
3. **A/B Test Interventions**: Test different retention strategies for different risk groups
4. **Expand Features**: Include additional data sources (support tickets, website behavior, etc.)
5. **Automate Actions**: Set up automated triggers for retention campaigns based on risk scores